# V19 sst2: official Kwon + K-prune (c 0.5) on the official DistilBERT-weights checkpoint, then d2i-RI, 5 seeds, frozen tag v19-freeze

In [ ]:
%%bash
set -e
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v19-freeze && git log --oneline -1 | tee /kaggle/working/commit.txt
export MODEL_NAME=distilbert-base-uncased BL_SKIP_CKPT=0 CK_REQUIRE=sst2 BL_TASKS=sst2 BL_CONS=0.5 PREP_ARGS=--no-ckpt
export CKPT_DIR_ROOT=/tmp/bl/ck_kprune/distilbert-base-uncased CKPT_SHA_FILE=$PWD/scripts/V19_CKPT_SHA256.json
bash scripts/v15_kaggle.sh bl /tmp/bl /kaggle/working/conf_sst2 || echo BL_FAILED
python -c "import transformers,torch;assert transformers.__version__=='4.51.3',transformers.__version__;print(torch.__version__,transformers.__version__)" | tee /kaggle/working/env.txt
export BEAT_ARGS="--methods d2i_ri --tasks sst2 --constraints 0.5 --seeds 7,42,77,123,2024 --kprune-masks /kaggle/working/conf_sst2/bl/kprune_masks.npz --ckpt-root $CKPT_DIR_ROOT"
bash scripts/v15_kaggle.sh beat /tmp/bl /kaggle/working/conf_sst2 || echo BEAT_FAILED
cd /kaggle/working && tar czf conf_sst2_results.tgz conf_sst2 commit.txt env.txt 2>/dev/null || true
ls -la /kaggle/working | head -20
